# CheXpert Dataset Exploration

Explore label distribution, frontal-view filtering, sample images, HuggingFace 4-class label encoding (`0=unlabeled, 1=uncertain, 2=absent, 3=present`), uncertainty policy sensitivity (`u_zeros` vs. `u_ones`), and 10k/20k subset stability for the **CheXpert** benchmark (`configs/chexpert.yaml`).

In [ ]:
"""
Colab environment setup: clone repository, install dependencies, mount Google Drive.
"""
import os
import subprocess

if os.path.basename(os.getcwd()) != "DinoV2_PEFT":
    if not os.path.exists("DinoV2_PEFT"):
        subprocess.run(["git", "clone", "https://github.com/RudhanLodhi/DinoV2_PEFT.git"], check=True)
    os.chdir("DinoV2_PEFT")

subprocess.run(["pip", "install", "-r", "requirements.txt", "-q"], check=True)

try:
    from google.colab import drive
    drive.mount("/content/drive")
    CHECKPOINT_DIR = "/content/drive/MyDrive/dinov2_peft"
except ImportError:
    CHECKPOINT_DIR = "outputs/checkpoints"

os.makedirs(CHECKPOINT_DIR, exist_ok=True)
print(f"Working directory:    {os.getcwd()}")
print(f"Checkpoint directory: {CHECKPOINT_DIR}")

## 1. Load Configuration & Dataset

Load `configs/chexpert.yaml` and `danjacobellis/chexpert` from HuggingFace Hub, inspect the 4-class integer label schema (`0=unlabeled, 1=uncertain, 2=absent, 3=present`), and apply frontal-only filtering matching our training notebooks.

In [ ]:
"""
Load CheXpert configuration and HuggingFace dataset; verify schema and frontal-view filter.
"""
from typing import Any
from datasets import load_dataset
from src.utils.config import load_config

cfg: dict[str, Any] = load_config("configs/chexpert.yaml")
COMPETITION_LABELS: list[str] = list(cfg["dataset"]["label_names"])
ALL_14_LABELS: list[str] = [
    "No Finding", "Enlarged Cardiomediastinum", "Cardiomegaly", "Lung Opacity",
    "Lung Lesion", "Edema", "Consolidation", "Pneumonia", "Atelectasis",
    "Pneumothorax", "Pleural Effusion", "Pleural Other", "Fracture", "Support Devices",
]

ds = load_dataset(cfg["dataset"]["hf_repo"])
print(ds)
print("\nSample pathology feature schema (Atelectasis):", ds["train"].features["Atelectasis"])
print("5 Competition Labels (configs/chexpert.yaml):", COMPETITION_LABELS)

# Build lightweight metadata-only DataFrames (drops PIL images to avoid OOM on Colab)
meta_train_df = ds["train"].remove_columns(["image"]).to_pandas()
meta_val_df = ds["validation"].remove_columns(["image"]).to_pandas()

if "Frontal/Lateral" in meta_train_df.columns:
    frontal_mask = meta_train_df["Frontal/Lateral"].isin([0, "Frontal", "frontal"])
    frontal_train_df = meta_train_df[frontal_mask].reset_index(drop=True)
else:
    frontal_train_df = meta_train_df

# Detect 4-class HF integer encoding (0=unlabeled, 1=uncertain, 2=absent, 3=present)
HF_4CLASS: bool = bool(cfg["dataset"].get("hf_4class_encoding", True))
POS_VAL = 3 if HF_4CLASS else 1.0
UNC_VAL = 1 if HF_4CLASS else -1.0
NEG_VAL = 2 if HF_4CLASS else 0.0

print(f"\nFull train rows:    {len(meta_train_df):,}")
print(f"Frontal train rows: {len(frontal_train_df):,}")
print(f"Official val rows:  {len(meta_val_df):,}")
print(f"Using 4-class HF encoding: {HF_4CLASS} (present={POS_VAL}, uncertain={UNC_VAL}, absent={NEG_VAL})")

## 2. Label Distribution (All 14 Labels & 5 Competition Benchmark Labels)

Compute positive prevalence and uncertainty frequencies using the true 4-class HuggingFace encoding, highlighting the 5 competition labels used across all experiments (`configs/chexpert.yaml`).

In [ ]:
"""
Compute positive occurrence rates and uncertainty counts per CheXpert pathology label.
"""
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

def compute_label_distribution(df: pd.DataFrame, label_cols: list[str]) -> pd.DataFrame:
    stats: list[dict[str, Any]] = []
    total_count: int = len(df)
    for col in label_cols:
        if col not in df.columns:
            continue
        series = df[col].fillna(0)
        pos_count: int = int((series == POS_VAL).sum())
        unc_count: int = int((series == UNC_VAL).sum())
        neg_count: int = total_count - pos_count - unc_count
        stats.append(
            {
                "label": col,
                "is_competition": col in COMPETITION_LABELS,
                "positive": pos_count,
                "uncertain": unc_count,
                "negative_or_unlabeled": neg_count,
                "pos_rate": pos_count / total_count,
                "unc_rate": unc_count / total_count,
            }
        )
    return pd.DataFrame(stats)

dist_df: pd.DataFrame = compute_label_distribution(frontal_train_df, ALL_14_LABELS)
print(dist_df[["label", "is_competition", "positive", "uncertain", "pos_rate", "unc_rate"]].to_string(index=False))

sns.set_theme(style="whitegrid")
fig, ax = plt.subplots(figsize=(12, 6))
colors = ["#d95f02" if c else "#4c72b0" for c in dist_df["is_competition"]]
sns.barplot(data=dist_df, x="pos_rate", y="label", palette=colors, ax=ax)
ax.set_title("CheXpert Positive Label Prevalence — Frontal Train Split (Orange = 5 Competition Labels)", fontsize=13, fontweight="bold")
ax.set_xlabel("Positive Rate (Present == 3)", fontsize=12)
ax.set_ylabel("Pathology Label", fontsize=12)
plt.tight_layout()
plt.show()

## 3. Sample Frontal Radiographs

Display a 4×4 visualization grid of frontal CheXpert radiographs annotated with their active positive competition findings.

In [ ]:
"""
Render a 4x4 grid of frontal CheXpert training images annotated with active positive findings.
"""
import matplotlib.pyplot as plt
from PIL import Image

def extract_positive_labels(example: dict[str, Any], label_cols: list[str]) -> list[str]:
    return [col for col in label_cols if example.get(col) == POS_VAL]

def display_chexpert_samples(dataset: Any, label_cols: list[str], num_samples: int = 16) -> None:
    fig, axes = plt.subplots(4, 4, figsize=(16, 16))
    shown = 0
    for idx in range(len(dataset)):
        if shown >= num_samples:
            break
        item = dataset[idx]
        if "Frontal/Lateral" in item and item["Frontal/Lateral"] not in (0, "Frontal", "frontal"):
            continue
        img = item["image"]
        if not isinstance(img, Image.Image):
            img = Image.open(img)
        positives: list[str] = extract_positive_labels(item, label_cols)
        title_text: str = "\n".join(positives[:3]) if positives else "No Comp. Finding"
        ax = axes.flat[shown]
        ax.imshow(img, cmap="gray")
        ax.set_title(title_text, fontsize=9)
        ax.axis("off")
        shown += 1
    plt.tight_layout()
    plt.show()

display_chexpert_samples(ds["train"], COMPETITION_LABELS, num_samples=16)

## 4. Uncertainty Policy Analysis (`u_zeros` vs. `u_ones`)

Compare effective positive prevalence across the 5 competition labels (and all 14 labels) under `u_zeros` (uncertain `1` → `0`) vs. `u_ones` (uncertain `1` → `1`).

In [ ]:
"""
Analyze uncertainty counts and compare effective positive rates under U-Zeros vs U-Ones policies.
"""
def compare_uncertainty_policies(df: pd.DataFrame, label_cols: list[str]) -> pd.DataFrame:
    records: list[dict[str, Any]] = []
    total_rows: int = len(df)
    for col in label_cols:
        if col not in df.columns:
            continue
        series = df[col].fillna(0)
        pos_count: int = int((series == POS_VAL).sum())
        unc_count: int = int((series == UNC_VAL).sum())
        u_zeros_rate: float = pos_count / total_rows
        u_ones_rate: float = (pos_count + unc_count) / total_rows
        records.append(
            {
                "label": col,
                "uncertain_count": unc_count,
                "u_zeros_rate": u_zeros_rate,
                "u_ones_rate": u_ones_rate,
                "rate_delta": u_ones_rate - u_zeros_rate,
            }
        )
    return pd.DataFrame(records)

policy_df: pd.DataFrame = compare_uncertainty_policies(frontal_train_df, COMPETITION_LABELS)
print(policy_df.to_string(index=False))

fig, ax = plt.subplots(figsize=(10, 5))
indices = np.arange(len(policy_df))
bar_h: float = 0.35
ax.barh(indices - bar_h / 2, policy_df["u_zeros_rate"], height=bar_h, label="U-Zeros (Active Protocol)", color="#2b5c8f")
ax.barh(indices + bar_h / 2, policy_df["u_ones_rate"], height=bar_h, label="U-Ones", color="#d95f02")
ax.set_yticks(indices)
ax.set_yticklabels(policy_df["label"], fontsize=11)
ax.set_xlabel("Effective Positive Prevalence", fontsize=12)
ax.set_title("CheXpert 5 Competition Labels: U-Zeros vs. U-Ones Policy Impact", fontsize=13, fontweight="bold")
ax.legend(loc="lower right", fontsize=11)
plt.tight_layout()
plt.show()

## 5. Train Subset Stability Analysis (10k vs. 20k vs. Full Frontal Split)

Verify that our default 10,000-sample subset (`seed=42`, with 20% deterministic held-out test carve-out) preserves pathology prevalence relative to the full frontal training split.

In [ ]:
"""
Verify label prevalence stability across 10k, 20k, and full frontal CheXpert splits.
"""
def evaluate_subset_distribution(df: pd.DataFrame, label_cols: list[str], sample_sizes: list[int], seed: int = 42) -> pd.DataFrame:
    records: list[dict[str, Any]] = []
    for sz in sample_sizes:
        tag: str = f"{sz // 1000}k" if sz < len(df) else "full_frontal"
        sub_df = df.sample(n=min(sz, len(df)), random_state=seed)
        for col in label_cols:
            if col in sub_df.columns:
                rate: float = float((sub_df[col].fillna(0) == POS_VAL).mean())
                records.append({"subset": tag, "label": col, "positive_rate": rate})
    return pd.DataFrame(records)

shift_df: pd.DataFrame = evaluate_subset_distribution(
    frontal_train_df, COMPETITION_LABELS, [10000, 20000, len(frontal_train_df)], seed=int(cfg["subset"]["seed"])
)
print(shift_df.pivot(index="label", columns="subset", values="positive_rate").round(4))

fig, ax = plt.subplots(figsize=(11, 5))
sns.barplot(data=shift_df, x="label", y="positive_rate", hue="subset", palette="Blues_d", ax=ax)
ax.set_title("CheXpert 5 Competition Labels: Prevalence Across Subset Sizes", fontsize=13, fontweight="bold")
ax.set_xlabel("Pathology Finding", fontsize=12)
ax.set_ylabel("Positive Prevalence (U-Zeros)", fontsize=12)
plt.xticks(rotation=20, ha="right")
plt.tight_layout()
plt.show()

## Summary

### Key Exploration Findings (Aligned with `configs/chexpert.yaml`)
- **4-Class Schema:** `danjacobellis/chexpert` uses integer ClassLabels (`0=unlabeled, 1=uncertain, 2=absent, 3=present`), handled consistently by `CheXpertDataset(hf_4class_encoding=True)`.
- **5 Competition Labels & Imbalance:** Across the frontal split, Pleural Effusion and Edema have the highest prevalence while Consolidation and Cardiomegaly are rarer — motivating the per-label `pos_weight` used in `04`–`10`.
- **Subsetting Stability:** The 10,000-sample frontal subset (`seed=42`, with 8,000 train / 2,000 deterministic held-out test + 234 official val) closely mirrors full-dataset prevalence across all 5 competition pathologies.